In [1]:
import pandas as pd
import re

# -------------------------------
# 1. Example: your clinical dataframe (adni_clinical)
# -------------------------------
# Assuming it's already loaded and has columns:
# ["PTID", "EXAMDATE", "DIAGNOSIS", "DXAD", ...]
# Convert EXAMDATE to datetime to make matching easier
adni_clinical = pd.read_csv("adni_data/MPR-R,_GradWarp,_B1,_N3,_Scaled_DXSUM_14Sep2025.csv")

adni_clinical["EXAMDATE"] = pd.to_datetime(adni_clinical["EXAMDATE"], errors="coerce")

# -------------------------------
# 2. Example: dataframe of your images
# -------------------------------
# Suppose you have a dataframe with filenames
image_df = pd.read_csv("subject_paths.csv")


In [2]:
import pandas as pd
import re

# -------------------------------
# 1. Example: your clinical dataframe (adni_clinical)
# -------------------------------
# Assuming it's already loaded and has columns:
# ["PTID", "EXAMDATE", "DIAGNOSIS", "DXAD", ...]
# Convert EXAMDATE to datetime to make matching easier
adni_gender = pd.read_csv("adni_data/All_Subjects_My_Table_11Aug2025.csv")

# -------------------------------
# 2. Example: dataframe of your images
# -------------------------------
# Suppose you have a dataframe with filenames
image_df = pd.read_csv("subject_paths.csv")


In [10]:
adni_gender

,subject_id,visit,PTGENDER,ARM,BCSTROKE,IHSYMPTOM,MHPSYCH,MH2NEURL,EXNEURO,EXNIMAG,EXILLNESS,EXLAB,CENROLL
0,941_S_10002,4_bl,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,941_S_10002,4_m12,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,941_S_10002,4_sc,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,941_S_10002,bl,2.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,941_S_10002,m12,2.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26335,037_S_4846,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26336,153_S_5238,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26337,068_S_5249,sc,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
26338,136_S_5001,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
image_df

,subject
0,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...
1,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_00...
2,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_02...
3,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_00...
4,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_00...
...,...
751,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_02...
752,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_02...
753,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_05...
754,adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_14...


In [7]:

# -------------------------------
# 3. Extract PTID and EXAMDATE from filenames
# -------------------------------
def parse_filename(fname):
    # Extract PTID (e.g., 002_S_0413)
    ptid_match = re.search(r"(\d{3}_S_\d{4})", fname)
    ptid = ptid_match.group(1) if ptid_match else None

    # Extract date (first 8 digits in that long number after 'Br_')
    date_match = re.search(r"Br_(\d{8})", fname)
    examdate = pd.to_datetime(date_match.group(1), format="%Y%m%d") if date_match else None

    return pd.Series([ptid, examdate])

image_df[["PTID", "EXAMDATE"]] = image_df["subject"].apply(parse_filename)
adni_gender.rename({"subject_id":"PTID"}, axis=1, inplace=True)
# -------------------------------
# 4. Merge with clinical dataframe
# -------------------------------
merged = pd.merge(
    image_df,
    adni_gender[["PTID", "PTGENDER"]],
    on=["PTID"], #, "EXAMDATE"],
    how="left"   # keep all images, even if diagnosis missing
)

# -------------------------------
# 5. Inspect results
# -------------------------------
print(merged.head())


                                             subject        PTID   EXAMDATE  \
0  adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...  011_S_0362 2010-10-29   
1  adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...  011_S_0362 2010-10-29   
2  adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...  011_S_0362 2010-10-29   
3  adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...  011_S_0362 2010-10-29   
4  adni_data/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_01...  011_S_0362 2010-10-29   

   PTGENDER  
0       2.0  
1       2.0  
2       2.0  
3       2.0  
4       2.0  


In [9]:
# Keep only one row per PTID
unique_gender = merged[["PTID", "PTGENDER"]].drop_duplicates(subset="PTID").reset_index(drop=True)

print(unique_gender)


           PTID  PTGENDER
0    011_S_0362       2.0
1    005_S_0448       1.0
2    023_S_0217       2.0
3    007_S_0344       1.0
4    005_S_0223       2.0
..          ...       ...
451  013_S_0575       2.0
452  011_S_0053       1.0
453  027_S_1277       1.0
454  053_S_0621       2.0
455  141_S_0697       1.0

[456 rows x 2 columns]


In [12]:
unique_gender['PTGENDER'].value_counts()

PTGENDER
1.0    253
2.0    203
Name: count, dtype: int64

In [2]:
import pandas as pd

# --- Ensure both are datetime ---
image_df["EXAMDATE"] = pd.to_datetime(image_df["EXAMDATE"])
adni_clinical["EXAMDATE"] = pd.to_datetime(adni_clinical["EXAMDATE"])

# --- Sort both by PTID and EXAMDATE ---
image_df = image_df.sort_values(["PTID", "EXAMDATE"]).reset_index(drop=True)
adni_clinical = adni_clinical.sort_values(["PTID", "EXAMDATE"]).reset_index(drop=True)

# --- Perform merge_asof ---
merged = pd.merge_asof(
    image_df,
    adni_clinical[["PTID", "EXAMDATE", "DIAGNOSIS", "DXAD"]],
    by="PTID",
    left_on="EXAMDATE",
    right_on="EXAMDATE",
    direction="backward"   # pick closest earlier exam
)

print(merged.head(10))


NameError: name 'image_df' is not defined